# Multi-Class Pneumonia Classification with Explainable AI (Grad-CAM)

## Notebook 03: Model Training

### Objectives

This notebook trains a deep learning model for multi-class pneumonia classification using the processed Chest X-ray dataset.

The main objectives are:

- Load the processed dataset
- Apply data augmentation
- Create DataLoaders
- Train the model
- Validate model performance
- Save the best model
- Record training history
- Generate training performance curves


## Import Required Libraries

In [1]:
# ============================================================
# Import Required Libraries
# ============================================================

import copy
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from sklearn.metrics import accuracy_score

sns.set_theme(style="whitegrid", context="notebook")

print("✅ Libraries imported successfully.")

✅ Libraries imported successfully.


## Project Paths


In [2]:
# ============================================================
# Project Paths
# ============================================================

PROJECT_ROOT = Path(
    "/mnt/g/Research paper/Research paper/Pneumonia-EfficientNetB0-XAI"
)

DATASET_DIR = PROJECT_ROOT / "dataset" / "processed_dataset"

MODELS_DIR = PROJECT_ROOT / "models"

RESULTS_DIR = PROJECT_ROOT / "results"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset : {DATASET_DIR}")
print(f"Models  : {MODELS_DIR}")
print(f"Results : {RESULTS_DIR}")

Dataset : /mnt/g/Research paper/Research paper/Pneumonia-EfficientNetB0-XAI/dataset/processed_dataset
Models  : /mnt/g/Research paper/Research paper/Pneumonia-EfficientNetB0-XAI/models
Results : /mnt/g/Research paper/Research paper/Pneumonia-EfficientNetB0-XAI/results


## Device Configuration

In [3]:
# ============================================================
# Device Configuration
# ============================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 60)
print(f"Device : {DEVICE}")

if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"CUDA   : {torch.version.cuda}")

print("=" * 60)

Device : cuda
GPU    : NVIDIA GeForce RTX 3050
CUDA   : 12.6


## Reproducibility

In [4]:
# ============================================================
# Random Seed
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

print(f"Random Seed : {SEED}")

Random Seed : 42


## Training Configuration

In [5]:
# ============================================================
# Training Configuration
# ============================================================

IMAGE_SIZE = 260          # Optimized for EfficientNet-B2
BATCH_SIZE = 16
EPOCHS = 30
LEARNING_RATE = 1e-4

NUM_CLASSES = 3
NUM_WORKERS = 4

CLASS_NAMES = [
    "BACTERIA",
    "NORMAL",
    "VIRUS"
]

print("=" * 60)
print(f"Image Size    : {IMAGE_SIZE}")
print(f"Batch Size    : {BATCH_SIZE}")
print(f"Epochs        : {EPOCHS}")
print(f"Learning Rate : {LEARNING_RATE}")
print("=" * 60)

Image Size    : 260
Batch Size    : 16
Epochs        : 30
Learning Rate : 0.0001


## Data Augmentation

Apply data augmentation to the training dataset and standard preprocessing to the validation and test datasets.

In [6]:
# ============================================================
# Data Transformations
# ============================================================

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.05, 0.05),
        scale=(0.95, 1.05)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

valid_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("✅ Data transformations created successfully.")

✅ Data transformations created successfully.


## Create Dataset Objects

Load the processed dataset using the ImageFolder class.

In [7]:
# ============================================================
# Load Datasets
# ============================================================

train_dataset = datasets.ImageFolder(
    root=DATASET_DIR / "train",
    transform=train_transform
)

validation_dataset = datasets.ImageFolder(
    root=DATASET_DIR / "validation",
    transform=valid_transform
)

test_dataset = datasets.ImageFolder(
    root=DATASET_DIR / "test",
    transform=valid_transform
)

print("✅ Dataset loaded successfully.")

✅ Dataset loaded successfully.
